# MiMo-V2.6-Flash-RL on CMP 170HX (SM80) — vLLM, pipeline parallel

| Metric | Value |
|---|---|
| Verdict | **SERVES — correct output, 3 cards, PP3** (measured 2026-09-23) |
| Decode, single stream, greedy (P1 median, 5 workloads) | **75.0 tok/s** PP3 · 68.3 tok/s PP4 |
| Decode, sampled T=1.0 (P2 median, warm) | **74.4 tok/s** PP3 · 67.7 tok/s PP4 |
| TTFT (short prompts, median) | 0.10 s PP3 · 0.11 s PP4 |
| Functional gate | PP3 4/4 correct + identical greedy · PP4 4/4 correct, 3/4 identical |
| KV cache (PP3, 16k context) | 73,202 tokens (4.5x concurrency at 16k) |
| Runtime | club fork image (PixelML/sm80vllm) + one model-file backport from upstream vLLM |

**PP3 on three cards beats PP4 on four** — one fewer pipeline hop, and it leaves the
x1-riser card out entirely. Three separate problems had to be fixed to get here; each is
documented below with its receipt.

```bash
docker pull ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905
```

In [ ]:
# --- Status cell ---
EXPERIMENT = "2026-09-22-mimo-v2.6-flash-4card-pp4-vllm"
RESULTS_DIR = "../results/" + EXPERIMENT
RECEIPTS = RESULTS_DIR + "/receipts"
LIVE = False

print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : serving. PP3 (3 cards) recommended; PP4 measured. Receipts: receipts/bench/{pp3,pp4}/")

In [ ]:
# --- Helpers: receipt loader and table renderer ---
import json, os, statistics
from IPython.display import display, Markdown


def receipt(*parts):
    path = os.path.join(RECEIPTS, *parts)
    with open(path) as fh:
        return json.load(fh)


def render_table(headers, rows):
    lines = ["| " + " | ".join(str(h) for h in headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown(chr(10).join(lines)))


assert not LIVE, "commit this notebook with LIVE = False"
print("receipts:", sorted(os.listdir(RECEIPTS)) if os.path.isdir(RECEIPTS) else "missing")

## 1. TL;DR

- **It runs.** MiMo-V2.6-Flash-RL (309B MoE, mxfp4 routed experts, fp8 block attention,
  attention sinks, DiffKV heads) serves correct, stable output on three CMP 170HX cards
  at 75 tok/s single-stream decode.
- **Why it was hard:** every production kernel this model ships for is SM90+. On SM80 each
  feature needs a fallback (section 3), and the one model-specific fallback that was
  wrong — fp8 QKV re-sharding for non-TP4 layouts — produced fluent-looking garbage.
- **Three fixes, all measured:** (1) a memory-unlock driver bug that crashed every big
  boot, (2) the upstream vLLM MiMo QKV-sharding fix backported onto the fork image,
  (3) a benchmark-tool bug that inflated decode rates on reasoning prompts.

In [ ]:
pins = {
    "model": "MiMo-V2.6-Flash-RL",
    "checkpoint": "XiaomiMiMo/MiMo-V2.6-Flash-RL",
    "checkpoint_revision": "5711b268169967567844e1e560e8a3966da959b1",
    "checkpoint_bytes": 172932505264,  # 65 shards, verified via tools/verify_checkpoint.py
    "quantization": "FP8 e4m3 block 128x128 dense/attention; routed experts MXFP4 (e8m0 block-32 scales)",
    "image": "ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905 (PixelML/sm80vllm @ 487ecf187)",
    "model_file_patch": "patches/mimo_v2.py = image file + upstream vLLM #57508 (211e252d0) + #57784 (9b2f34cad)",
    "attention_backend": "TRITON_ATTN_DIFFKV (auto-selected; sinks + DiffKV on SM80)",
    "moe_backend": "MARLIN Mxfp4 (W4A16 dequant; auto-selected)",
    "dense_fp8": "MarlinFP8ScaledMMLinearKernel (weight-only fp8, no fp8 tensor cores on SM80)",
    "topology_recommended": "PP3 on the three x8/x16 cards, VLLM_PP_LAYER_PARTITION=17,16,15",
    "cards": "CMP 170HX (GA100, SM80, 64 GiB after memory unlock), Gen1 links x8/x1/x16/x16, no NVLink, no P2P",
    "driver": "open kernel modules 610.43.03 + cmpunlocker, with patches/cmpunlocker-late-pma-wprfix.diff",
    "power_cap": "180 W bench cap",
    "measured_utc": {"pp4": "2026-09-23T18:16:11Z", "pp3": "2026-09-23T18:40:04Z"},
}
for k, v in pins.items():
    print(f"{k:22s} {v}")

## 2. Results (measured)

Protocol mirrors the 2026-09-05 GLM-5.3-Flash lane. **P1:** greedy, 512 output
tokens, 5 workloads x 3 reps, median. **P2:** T=1.0 / top_p 0.95 (vendor
recommendation), `ignore_eos`, 5 reps, first rep dropped as cold. Token counts come
only from the final streamed `usage` object. Reasoning tokens count as generated
tokens (see fix 3). Single request at a time; no speculative decode.

In [ ]:
rows = []
for topo in ('pp3', 'pp4'):
    p1 = receipt('bench', topo, 'p1.json')
    p2 = receipt('bench', topo, 'p2.json')
    per = {w: statistics.median(r['decode_tok_s'] for r in v['reps']) for w, v in p1['runs'].items()}
    allr = [r for v in p1['runs'].values() for r in v['reps']]
    warm = p2['runs']['longform']['reps'][1:]
    rows.append([topo.upper(), *[f"{per[w]:.1f}" for w in ('code', 'json', 'counting', 'math', 'prose')],
                 f"{statistics.median(r['decode_tok_s'] for r in allr):.1f}",
                 f"{statistics.median(r['decode_tok_s'] for r in warm):.1f}",
                 f"{statistics.median(r['ttft_s'] for r in allr):.3f}",
                 sum(r['degenerate'] for r in allr)])
render_table(['Topology', 'code', 'json', 'counting', 'math', 'prose', 'P1 median', 'P2 median', 'TTFT s', 'degenerate'], rows)

In [ ]:
rows = []
for topo in ('pp3', 'pp4'):
    for g in receipt('bench', topo, 'gate.json'):
        rows.append([topo.upper(), g['prompt'][:44], g['answers'][0][:48].replace('|', '/'), g['correct'], g['identical']])
render_table(['Topology', 'Prompt', 'Answer (rep 1)', 'Correct 3/3', 'Identical 3/3'], rows)

Greedy repeats are not bit-identical on this stack in general (the GLM lane measured
the same drift): Marlin MoE uses atomic-add reductions and the Triton DiffKV
split-softmax path changes accumulation order. Every answer above is correct; the one
PP4 non-identical case differs in wording only.

**Untested:** concurrency sweep, long-prompt prefill sweep, sustained stability run,
and the shipped MTP / DFlash drafters. These are the next measurements.

## 3. Why this model is hard on SM80

| Model feature | Production kernel | SM80 path used here |
|---|---|---|
| Attention sinks on SWA layers | FlashAttention 3 (SM90+) | Triton DiffKV backend (implements sinks) |
| DiffKV heads (QK 192 / V 128) | FA-DiffKV (SM90+) | Triton DiffKV; head 192 padded to 256; bf16 KV only |
| MXFP4 routed experts | TRT-LLM / CUTLASS / DeepGEMM (SM90/100) | Marlin W4A16 dequant, experts repacked at load |
| FP8 block dense + attention | FP8 tensor cores (SM89+) | Marlin weight-only FP8 |
| Fused QKV pre-sharded for TP4 | the checkpoint's own TP4 layout | PP needs TP1 re-sharding: dequant, reorder, requant (fix 2) |
| 161 GiB of weights | 8x80 GB or 4x H200 | 3x64 GiB PP3 at 0.97 utilization, or 4-card PP4 |

The official recipe runs TP4. On this fabric TP is not an option (no P2P, Gen1
links, one x1 riser — TP measured 6.6x worse in the GLM lane), so pipeline
parallel forces the TP1 re-sharding path that almost nobody else exercises.

## 4. The three fixes

### Fix 1 — memory-unlock driver handed out firmware-protected VRAM (every earlier crash)

The 64 GiB unlock (cmpunlocker `late-pma.patch`) registers the top reserved
framebuffer region `0xff7300000-0xfffffffff` (141 MB) with the physical memory
allocator. After the unlock relocates it, the GSP firmware's write-protected region
(WPR) sits inside that range (`wprStart=0xff7400000 wprEnd=0xffff00000`, logged at
boot). Any allocation that lands there faults with **Xid 31 MMU REGION_VIOLATION**,
and the driver's own memory scrubber then faults on the same pages and wedges the
GPUs until a reboot.

Only near-full cards touch that range, which is why the fault looked random, why
small standalone repros always passed, and why it showed up inside the Marlin expert
repack (the largest allocation burst of the boot). Receipt: attempts A6-A9, A10.

Fix: skip registering that region (`patches/cmpunlocker-late-pma-wprfix.diff`,
costs 141 MB per card). After the fix all four cards filled 63.0 GiB and verified every
chunk with zero Xid, and every MiMo boot since has passed the repack.

### Fix 2 — fp8 QKV re-sharding: backport the upstream vLLM fix

The checkpoint stores fused `qkv_proj` as 4 chunks of `[Q|K|V]` with per-chunk fp8
block scales (layer 0: weight `[13568,4096]`, scale `[108,32]` = 4 x 27). Under
PP the model runs at TP1, and the fork's `mimo_v2.py` must dequantize, reorder and
requantize. Our first local patch sliced scales as if they tiled the whole tensor,
corrupting every attention layer: fluent-looking garbage (`2,,,,,,`). Upstream
vLLM fixed exactly this in #57508 (`ckpt_tp = num_key_value_heads`, per-chunk scale
tiling); #57784 moves the MoE router to fp32 logits. Both apply cleanly to the
image's file: `patches/mimo_v2.py`. It is the only runtime patch.

### Fix 3 — the benchmark tool must count reasoning tokens

MiMo streams its thinking as `reasoning_content`. The first version of
`tools/bench_mimo.py` started TTFT at the first *content* token, so on reasoning-heavy
prompts decode time shrank and rates read as 180-350 tok/s. It now times from the first
token of either kind; all numbers above use the fixed tool.

In [ ]:
attempts = receipt('runtime_attempts.json')
rows = []
for a in attempts['attempts']:
    rows.append([a["id"], a.get("topology", "-")[:48], a["result"][:80]])
render_table(["Attempt", "Topology", "Result"], rows)

## 5. Reproduce

### Driver (unlocked 170HX only)

Apply `patches/cmpunlocker-late-pma-wprfix.diff` to the cmpunlocker build tree,
rebuild the modules, reboot. Verify in the kernel log:

```bash
journalctl -k -b 0 | grep pixelml-wprfix   # one line per card
journalctl -k -b 0 | grep -c Xid            # 0
```

### Weights (NVMe strongly preferred: ~2.5 min load vs ~35 min from HDD-class NFS)

```bash
hf download XiaomiMiMo/MiMo-V2.6-Flash-RL --revision 5711b268169967567844e1e560e8a3966da959b1 --local-dir /models/mimo-v2.6-flash-rl
python3 results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/tools/verify_checkpoint.py /models/mimo-v2.6-flash-rl
```

### Launch — PP3 on the three wide-link cards (recommended)

Pick the three cards that are not on the x1 riser (`nvidia-smi --query-gpu=index,pcie.link.width.current --format=csv`).

```bash
P=$PWD/results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/patches
docker run -d --name mimo --gpus '"device=0,2,3"' --shm-size 32g \
  -e VLLM_PP_LAYER_PARTITION=17,16,15 \
  -v /models:/models \
  -v $P/mimo_v2.py:/usr/local/lib/python3.12/dist-packages/vllm/model_executor/models/mimo_v2.py:ro \
  -p 8000:8000 ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-pp-20260905 \
  --model /models/mimo-v2.6-flash-rl --served-model-name mimo-v2.6-flash \
  --pipeline-parallel-size 3 --trust-remote-code \
  --gpu-memory-utilization 0.97 --max-model-len 16384 --max-num-seqs 16 \
  --reasoning-parser mimo
```

Expect `Using TRITON_ATTN_DIFFKV for attention` and `Using 'MARLIN' Mxfp4 MoE backend`
in the log. The default 16/16/16 split leaves the last rank (which also holds the
`lm_head`) with no KV memory; 17/16/15 balances it.

### Launch — PP4 on all four cards

Same command with `--gpus all`, `--pipeline-parallel-size 4`, no partition variable,
`--gpu-memory-utilization 0.85 --max-model-len 32768`. Slower here (68 tok/s): the
extra hop crosses the x1 card.

### Gate and benchmark

```bash
python3 results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/tools/gate.py gate.json
python3 results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/tools/bench_mimo.py --protocol p1 --out p1.json
python3 results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/tools/bench_mimo.py --protocol p2 --out p2.json
```

### Attribution

- Model: Xiaomi MiMo team, MiMo-V2.6 release (MIT), 2026-09-22.
- MiMo QKV-sharding and router fixes: upstream vLLM #57508 and #57784.
- Memory unlock: cmpunlocker (open-source); the WPR fix is this lane's.
- PP-on-170HX shape and protocol: this club's DeepSeek-V4-Flash and GLM-5.3-Flash lanes.
- PP3 suggestion: the PR #57 thread.

## 6. Try your own prompt

Edit the message and run against the server from section 5.

In [ ]:
%%bash
curl -s http://localhost:8000/v1/chat/completions -H 'Content-Type: application/json' -d '{
  "model": "mimo-v2.6-flash",
  "messages": [{"role": "user", "content": "Edit me before sending."}],
  "max_tokens": 1024,
  "temperature": 1.0, "top_p": 0.95
}' | python3 -c 'import json,sys; d=json.load(sys.stdin); m=d["choices"][0]["message"]; print(m.get("content")); print(d["usage"])'